# gpt-oss-20b 环境检查

这个 notebook 用来确认当前环境是否适合运行 gpt-oss-20b 的训练和部署流程。

gpt-oss-20b 和普通 dense 模型不完全一样。它涉及 MoE、Harmony 格式、reasoning effort、MXFP4，以及不同后端对这些能力的支持差异。环境检查必须比普通模型更谨慎。

## 1. Python、系统和 GPU

先确认 Python、系统、PyTorch、CUDA 和 GPU 显存。gpt-oss-20b 虽然 active parameters 低于总参数，但总权重加载仍然需要足够内存或合适的量化/压缩支持。

In [ ]:
import os
import platform
import sys
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {props.name}, {props.total_memory / 1024**3:.1f} GB")

## 2. 核心包版本

重点检查：

- `transformers`：是否支持 gpt-oss 架构和 Harmony 相关处理。
- `trl`：SFT/DPO/PPO/GRPO 训练器。
- `peft`：LoRA/QLoRA。
- `bitsandbytes`：低显存加载。
- `vllm`：服务端推理。
- `unsloth`：gpt-oss 训练和运行支持。

In [ ]:
from importlib.metadata import PackageNotFoundError, version

packages = ["transformers", "datasets", "trl", "peft", "accelerate", "bitsandbytes", "vllm", "unsloth"]
for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

## 3. 部署命令检查

gpt-oss-20b 的本地运行优先检查 Ollama，因为 `gpt-oss:20b` 有官方 Ollama 路线。服务化优先检查 vLLM。SGLang 是否完整支持要以当前版本为准。

In [ ]:
import shutil

for command in ["ollama", "vllm", "sglang", "llama-cli", "llama-server"]:
    print(f"{command}: {shutil.which(command) or 'not found'}")

## 4. 判断结论

- 只想本地运行：优先确认 Ollama 可用。
- 想服务化：确认 vLLM 版本支持 gpt-oss。
- 想训练：优先使用 LoRA/QLoRA/Unsloth，不建议新人直接全量微调。
- 看到 fp16、MXFP4、Harmony 相关错误时，不要按普通 Qwen 模型思路处理，要先检查模板、精度和框架版本。